# Reproduce the EU27 material–climate publication files

Run this notebook in a fresh runtime. Upload the analysis package from the first notebook. The notebook checks out the recorded Git commit, verifies the analysis, reproduces the manuscript and Supplementary Information tables and figures, compares them with the checked-in reference publication files, and downloads one publication-results ZIP.


In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly eu27-material-climate-analysis.zip")
analysis_zip = Path("/content") / next(iter(uploaded))
with zipfile.ZipFile(analysis_zip) as z:
    bad = z.testzip()
    if bad is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad}")
    member = "eu27-material-climate-analysis/run_receipt.json"
    if member not in z.namelist():
        raise RuntimeError("Analysis package does not contain run_receipt.json")
    receipt = json.loads(z.read(member).decode("utf-8"))
if receipt.get("mode") != "public_repository" or not receipt.get("git_commit"):
    raise RuntimeError("Analysis package does not contain the recorded Git commit and repository manifest")
print("Recorded Git commit:", receipt["git_commit"])

In [ ]:
import hashlib, os
REPOSITORY_URL = "https://github.com/calinadriancomes/eu27-material-climate-sensitivity.git"
repo = Path("/content/eu27-material-climate-sensitivity")
repo.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "init"], cwd=repo, check=True, stdout=subprocess.DEVNULL)
subprocess.run(["git", "remote", "add", "origin", REPOSITORY_URL], cwd=repo, check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", receipt["git_commit"]], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True, stdout=subprocess.DEVNULL)
os.chdir(repo)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if actual_commit != receipt["git_commit"]:
    raise RuntimeError("Checked-out Git commit does not match the analysis receipt")
manifest_sha = hashlib.sha256(Path("checksums.sha256").read_bytes()).hexdigest()
if manifest_sha != receipt.get("repository_manifest_sha256"):
    raise RuntimeError("Repository manifest does not match the analysis receipt")
print("Repository commit verified")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-lock.txt"], check=True)
subprocess.run([sys.executable, "code/checks.py", "verify"], check=True)
sys.path.insert(0, str(repo / "code"))
from checks import safe_extract
subprocess.run([sys.executable, "code/checks.py", "verify-external", str(analysis_zip), "eu27-material-climate-analysis"], check=True)
extract_dir = repo / "work" / "stage2_input"
safe_extract(analysis_zip, extract_dir)
analysis_root = extract_dir / "eu27-material-climate-analysis"

In [ ]:
import shutil
work_analysis = repo / "work" / "analysis"
if work_analysis.exists():
    shutil.rmtree(work_analysis)
shutil.copytree(analysis_root / "numeric", work_analysis)
shutil.copy2(analysis_root / "run_receipt.json", repo / "work" / "analysis_receipt.json")
subprocess.run([sys.executable, "code/checks.py", "compare-analysis"], check=True)
# Recompute the adopted secondary publication evidence from the packaged frozen panel.
subprocess.run([
    sys.executable, "code/05_build_extended_publication_evidence.py",
    "--panel", str(analysis_root / "data" / "panel_2010_2023.csv"),
    "--analysis", "work/analysis",
], check=True)

# Reproduce the manuscript and Supplementary Information publication layer in a fresh output directory.
publication_root = repo / "work" / "publication"
if publication_root.exists():
    shutil.rmtree(publication_root)
subprocess.run([
    sys.executable, "code/04_render_publication_figures.py",
    "--analysis", "work/analysis",
    "--output-root", "work/publication",
    "--natural-earth", "cartography/source/ne_10m_admin_0_countries.zip",
], check=True)
subprocess.run([sys.executable, "code/checks.py", "compare-publication", "--publication", "work/publication"], check=True)
publication_output = Path("/content/eu27-material-climate-publication-results.zip")
subprocess.run([
    sys.executable, "code/checks.py", "package-publication",
    "--publication", "work/publication",
    "--output", str(publication_output),
], check=True)
subprocess.run([sys.executable, "code/checks.py", "verify"], check=True)

print("Final publication layer reproduced and matched the checked-in reference files with 0 differences.")
files.download(str(publication_output))
